# Simple neural network model

This notebook trains a simple scikit-learn multilayer perceptron for the five-class delay prediction task. The hidden layers use ReLU activation and the multiclass output uses softmax probabilities. The chronological test partition remains untouched.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.neural_network import MLPClassifier

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model_data import prepare_model_data
from model_evaluation import macro_f1_test

## Prepare a reproducible sample

The shared preprocessing is fitted on training rows only. A sample is used because fitting a neural network on millions of sparse rows is computationally expensive for an initial experiment.

In [ ]:
model_data = prepare_model_data(PROJECT_ROOT / "data")

train_size = min(500_000, model_data.X_train.shape[0])
validation_size = min(250_000, model_data.X_validation.shape[0])

X_train = model_data.X_train[:train_size]
y_train = model_data.y_train.iloc[:train_size]
X_validation = model_data.X_validation[:validation_size]
y_validation = model_data.y_validation.iloc[:validation_size]

pd.DataFrame({
    "partition": ["train", "validation"],
    "rows": [len(y_train), len(y_validation)],
    "features": [X_train.shape[1], X_validation.shape[1]],
})

## Train the neural network

The candidates use ReLU hidden layers and a softmax output for multiclass targets. A small validation-based comparison varies network width, regularisation, and learning rate; the chronological validation partition is reserved for model selection.

In [4]:
neural_network_candidates = {
    "neural_network_wider": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size=1024,
        learning_rate_init=0.001,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        max_iter=100,
        random_state=42,
    ),
    "neural_network_regularized": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size=1024,
        learning_rate_init=0.001,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        max_iter=100,
        random_state=42,
    ),
    "neural_network_lower_rate": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size=1024,
        learning_rate_init=0.0003,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        max_iter=100,
        random_state=42,
    ),
}

neural_network_results = []
for name, model in neural_network_candidates.items():
    model.fit(X_train, y_train)
    neural_network_results.append(
        macro_f1_test(
            model,
            X_validation,
            y_validation,
            model_name=name,
        )
    )

neural_network_results = (
    pd.DataFrame(neural_network_results)
    .sort_values(["macro_f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
neural_network_results

,model,n_test,macro_f1,accuracy
0,neural_network_wider,250000,0.263354,0.382544
1,neural_network_regularized,250000,0.262863,0.376892
2,neural_network_lower_rate,250000,0.260212,0.382148


## Exploratory result

The neural network is retained as an exploratory experiment but is not included in the main model comparison. Its initial validation macro-F1 was lower than logistic regression and LightGBM, while training wasn't computationally cheaper. The final workflow therefore focuses on logistic regression and LightGBM.